# Connected-component labeling in raster masks

Compare 4- and 8-neighbour connectivity and inspect how invalid cells split foreground regions.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/raster-components'))
from connected_components import connected_components


## 1. Compare connectivity rules

The foreground mask contains diagonal contacts. Four-connectivity uses shared edges; eight-connectivity also includes diagonal neighbours.

In [ ]:
mask=np.array([[1,0,1,0,0],[0,1,0,0,1],[1,0,1,1,1],[0,0,0,1,0]],dtype=bool)
labels4,sizes4=connected_components(mask,connectivity=4)
labels8,sizes8=connected_components(mask,connectivity=8)
fig,axes=plt.subplots(1,3,figsize=(11,4))
for ax,data,title in zip(axes,[mask,labels4,labels8],['Foreground','4-connectivity','8-connectivity']):
    im=ax.imshow(data,cmap='tab20'); ax.set_title(title); fig.colorbar(im,ax=ax,shrink=.75)
plt.tight_layout(); plt.show()
print('4-connected component sizes:',sizes4)
print('8-connected component sizes:',sizes8)


## 2. Invalid cells are barriers

A validity mask can exclude NoData cells. They are not background connections; they are removed from the graph and remain label zero.

In [ ]:
valid=np.ones(mask.shape,dtype=bool); valid[:,2]=False
labels_barrier,sizes_barrier=connected_components(mask,connectivity=8,valid=valid)
plt.figure(figsize=(5,4)); plt.imshow(labels_barrier,cmap='tab20'); plt.title('Labels with invalid barrier column'); plt.colorbar(); plt.show()
print('Component sizes:',sizes_barrier)


## Investigate

1. Which regions merge under 8-connectivity?
2. What happens when a NoData barrier cuts across a region?
3. Why are component IDs arbitrary rather than stable feature identifiers?
4. How would a tiled implementation reconcile components crossing tile boundaries?